In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
# LLM 선언 - AzureOpenAI 로 활용해도 됨

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini") 
small_llm = ChatOpenAI(model="gpt-4o-mini")

In [ ]:
# node
# 1. market research
# 2. 주식회사                          ---- [최종적으로 투자?]
# 3. 재무제표 - yfinance

In [3]:
# %pip install --upgrade --quiet yfinance

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
from langchain_community.tools.yahoo_finance_news import YahooFinanceNewsTool
# from langgraph.prebuilt import create_react_agent # deprecated 
from langchain.agents import create_agent
# 스스로 어떤 도구를 쓸지 판단해서 결과를 준다

from langgraph.graph import MessagesState
from langchain_core.messages import HumanMessage
from langgraph.types import Command
from typing import Literal

market_research_agent = create_agent(
    llm, 
    tools=[YahooFinanceNewsTool()], 
    system_prompt = 'You are a market researcher. Provide fact only not opinions'
)


def market_research_node(state: MessagesState) -> Command[Literal["supervisor"]]:
    result = market_research_agent.invoke(state)  # 실행 결과
    print(f'market_research_node: {result}')
    return Command(
        update = {'messages' : [HumanMessage(content=result['messages'][-1].content, name='market_research')]},
        goto='stock-research'
    )

In [6]:
from langchain.tools import tool
import yfinance as yf

@tool
def get_stock_price(ticker:str) -> str:
    """ given a stock ticker, return the price data for the past 10 days """
    
    stock_info = yf.download(ticker, period='10d').to_dict()
    return stock_info

stock_research_tools = [get_stock_price]
stock_research-agent = create_agent(
    llm,
    tools=stock_research_tools,
    system_prompt='You are a market researcher. Provide fact only not opinions'
)






In [7]:
get_stock_price('AAPL')

[*********************100%***********************]  1 of 1 completed


{('Close', 'AAPL'): {Timestamp('2026-03-09 00:00:00'): 259.8800048828125,
  Timestamp('2026-03-10 00:00:00'): 260.8299865722656,
  Timestamp('2026-03-11 00:00:00'): 260.80999755859375,
  Timestamp('2026-03-12 00:00:00'): 255.75999450683594,
  Timestamp('2026-03-13 00:00:00'): 250.1199951171875,
  Timestamp('2026-03-16 00:00:00'): 252.82000732421875,
  Timestamp('2026-03-17 00:00:00'): 254.22999572753906,
  Timestamp('2026-03-18 00:00:00'): 249.94000244140625,
  Timestamp('2026-03-19 00:00:00'): 248.9600067138672,
  Timestamp('2026-03-20 00:00:00'): 247.99000549316406},
 ('High', 'AAPL'): {Timestamp('2026-03-09 00:00:00'): 261.1499938964844,
  Timestamp('2026-03-10 00:00:00'): 262.4800109863281,
  Timestamp('2026-03-11 00:00:00'): 262.1300048828125,
  Timestamp('2026-03-12 00:00:00'): 258.95001220703125,
  Timestamp('2026-03-13 00:00:00'): 256.3299865722656,
  Timestamp('2026-03-16 00:00:00'): 253.88999938964844,
  Timestamp('2026-03-17 00:00:00'): 255.1300048828125,
  Timestamp('2026-0